In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import os

df = pd.read_csv(os.path.join(path, 'Q3_data.csv'))

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
for col in df.columns:
  df[col] = df[col].fillna( df[col].mean() )

df

In [ ]:
print(df.duplicated().sum())
df = df.drop_duplicates()

In [ ]:
df.select_dtypes(include=['object']).columns

# There isn't any

In [ ]:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()


In [ ]:
from sklearn.model_selection import train_test_split

X,y = df.drop('Target',axis=1), df['Target']

# split ratio (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,        # 20% for test, remaining 80% for train
    random_state=42,      # reproducible output
    shuffle=True,         # representative splits
)


In [ ]:
from IPython.display import clear_output
%pip install catboost -q
clear_output()

from sklearn.metrics import f1_score,accuracy_score
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostClassifier

import numpy as np


cb_model = CatBoostClassifier(random_state=42, verbose=0)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

f1_scores_averaged = []
acc_scores_averaged = []

for train_idx, val_idx in kfold.split(X_train):
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    cb_model.fit(X_fold_train, y_fold_train)

    cb_preds = cb_model.predict(X_fold_val)

    f1_scores_averaged.append(f1_score(y_fold_val, cb_preds))
    acc_scores_averaged.append(accuracy_score(y_fold_val, cb_preds))

print(f"[+] CatBoost \nAvg F1: {np.mean(f1_scores_averaged)}\nAvg Accuracy: {np.mean(acc_scores_averaged)}")

In [ ]:
import matplotlib.pyplot as plt

# Feature importance
feature_importance = pd.DataFrame({
    'feature': df.drop('Target',axis=1).columns,
    'importance': cb_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

# Something Strange Xd :)

In [ ]:
golden_strange_weird_feature_idx = list(cb_model.feature_importances_).index(max(cb_model.feature_importances_))
print('The weird feature is :',df.drop('Target',axis=1).columns[0])

In [ ]:
# Task Bonus: Write your code here: